In [6]:
import os
import numpy as np
import pandas as pd
from scipy.stats import weightedtau

def compare_to_full(full_csv, reduced_csv):

    full = pd.read_csv(full_csv)
    reduced = pd.read_csv(reduced_csv)

    merged = full.merge(
        reduced,
        on="Train_ID",
        suffixes=("_full", "_reduced"),
        validate="one_to_one"
    )

    full_scores = merged["Score_full"].to_numpy()
    reduced_scores = merged["Score_reduced"].to_numpy()

    # Weighted Kendall Tau
    wtau, _ = weightedtau(full_scores, reduced_scores)

    # Top-10% overlap
    k = int(np.ceil(len(merged) * 0.10))

    top_full = set(
        merged.nlargest(k, "Score_full")["Train_ID"]
    )

    top_reduced = set(
        merged.nlargest(k, "Score_reduced")["Train_ID"]
    )

    overlap = len(top_full & top_reduced) / k

    return wtau, overlap

In [7]:
runs = [
    "RealRun1",
    "RealRun2",
    "RealRun3"
]

settings = [
    "80per",
    "60per",
    "40per",
    "20per",
    "10per",
    "05per"
]

results = []

for run in runs:

    full_file = os.path.join(
        run,
        "TC_Train_Set_Checkpoints_Full.csv"
    )

    for setting in settings:

        reduced_file = os.path.join(
            run,
            f"TC_Train_Set_Checkpoints_{setting}.csv"
        )

        wtau, overlap = compare_to_full(
            full_file,
            reduced_file
        )

        results.append({
            "Run": run,
            "Checkpoint": setting,
            "Weighted Kendall Tau": wtau,
            "Top10% Overlap": overlap
        })

results = pd.DataFrame(results)

print(results)

         Run Checkpoint  Weighted Kendall Tau  Top10% Overlap
0   RealRun1      80per              0.994268         0.98125
1   RealRun1      60per              0.990531         0.97000
2   RealRun1      40per              0.986814         0.95000
3   RealRun1      20per              0.976642         0.91875
4   RealRun1      10per              0.962952         0.87125
5   RealRun1      05per              0.940586         0.78875
6   RealRun2      80per              0.994268         0.98125
7   RealRun2      60per              0.990531         0.97000
8   RealRun2      40per              0.986814         0.95000
9   RealRun2      20per              0.976642         0.91875
10  RealRun2      10per              0.962952         0.87125
11  RealRun2      05per              0.940586         0.78875
12  RealRun3      80per              0.994268         0.98125
13  RealRun3      60per              0.990531         0.97000
14  RealRun3      40per              0.986814         0.95000
15  Real

In [8]:
summary = (
    results
    .groupby("Checkpoint")
    .agg({
        "Weighted Kendall Tau": ["mean", "std"],
        "Top10% Overlap": ["mean", "std"]
    })
)

print(summary)

           Weighted Kendall Tau      Top10% Overlap     
                           mean  std           mean  std
Checkpoint                                              
05per                  0.940586  0.0        0.78875  0.0
10per                  0.962952  0.0        0.87125  0.0
20per                  0.976642  0.0        0.91875  0.0
40per                  0.986814  0.0        0.95000  0.0
60per                  0.990531  0.0        0.97000  0.0
80per                  0.994268  0.0        0.98125  0.0


In [9]:
# from scipy.stats import kendalltau, weightedtau
# import pandas as pd
# import numpy as np

# def compare_two_runs(file_a, file_b):
#     a = pd.read_csv(file_a)
#     b = pd.read_csv(file_b)

#     merged = a.merge(
#         b,
#         on="Train_ID",
#         suffixes=("_run1", "_real"),
#         validate="one_to_one"
#     )

#     score_a = merged["Score_run1"].to_numpy()
#     score_b = merged["Score_real"].to_numpy()

#     # Ordinary Kendall Tau
#     kendall, kendall_p = kendalltau(
#         score_a,
#         score_b,
#         variant="b"
#     )

#     # Weighted Kendall Tau
#     weighted, _ = weightedtau(
#         score_a,
#         score_b
#     )

#     # Top-10% overlap
#     k = max(1, int(np.ceil(len(merged) * 0.10)))

#     top_a = set(
#         merged.nlargest(k, "Score_run1")["Train_ID"]
#     )

#     top_b = set(
#         merged.nlargest(k, "Score_real")["Train_ID"]
#     )

#     top10_overlap = len(top_a & top_b) / k

#     # Sign agreement
#     sign_agreement = np.mean(
#         np.sign(score_a) == np.sign(score_b)
#     )

#     return pd.Series({
#         "Number of matched samples": len(merged),
#         "Kendall Tau": kendall,
#         "Kendall p-value": kendall_p,
#         "Weighted Kendall Tau": weighted,
#         "Top-10% Overlap": top10_overlap,
#         "Sign Agreement": sign_agreement
#     })

In [10]:
# result = compare_two_runs(
#     "Run1/TC_Train_Set_Checkpoints_Full.csv",
#     "RealRun/TC_Train_Set_Checkpoints_Full.csv"
# )

# print(result)